In [ ]:
#@title Video Studio voice engine v3: press ▶ (Run), then allow Google Drive
# VIDEO STUDIO VOICE ENGINE - press Run (▶) and allow Google Drive. Nothing to change here.
# Turn on the graphics card first: Runtime > Change runtime type > T4 GPU.
"""Video Studio voice engine (UPGRADE v49) - makes custom voices with Qwen3-TTS.

The SAME file runs in two places:
  * Google Colab (fast, free GPU): Video Studio turns it into a one-cell notebook. You press Run once, allow
    Google Drive, and it works through the voice jobs Video Studio puts in your Drive folder "Video Studio voices".
  * This computer (no Colab, slower): Video Studio starts it by itself in the background.
Voices are cloned the way Qwen recommends for the closest match (accent, rhythm, tone): the voice's sample AND the
exact words said in it ("ICL" mode). The voice is prepared once and reused for every line of the script, so a whole
video keeps one consistent voice. Nothing is kept anywhere except your own Drive / your own computer.
"""
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import traceback
import wave

WORKER_VERSION = 3
FOLDER = "Video Studio voices"
BASE_GPU = "Qwen/Qwen3-TTS-12Hz-1.7B-Base"
BASE_CPU = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
DESIGN = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
HEARTBEAT = 15


# ---------------------------------------------------------------- small helpers
def _rj(p, default=None):
    try:
        with open(p, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def _wj(p, data):
    tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1)
    os.replace(tmp, p)


def _md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def _write_wav(path, audio, sr):
    import numpy as np
    a = np.asarray(audio, dtype=np.float32).reshape(-1)
    pcm = (np.clip(a, -1, 1) * 32767).astype(np.int16)
    tmp = path + ".part"
    with wave.open(tmp, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(int(sr))
        w.writeframes(pcm.tobytes())
    os.replace(tmp, path)


class Screen:
    """One big status line: in Colab it updates in place; on this computer it prints."""

    def __init__(self, colab):
        self.colab, self.h = colab, None

    def show(self, icon, text, small=""):
        if self.colab:
            from IPython.display import HTML, display
            html = (f"<div style='font:600 22px/1.35 system-ui,Segoe UI,Arial;padding:14px 18px;border-radius:12px;"
                    f"background:#111827;color:#F9FAFB'>{icon} {text}"
                    + (f"<div style='font:400 14px/1.4 system-ui;opacity:.75;margin-top:6px'>{small}</div>" if small else "")
                    + "</div>")
            if self.h is None:
                self.h = display(HTML(html), display_id=True)
            else:
                self.h.update(HTML(html))
        else:
            print(f"{icon} {text}" + (f"  ({small})" if small else ""), flush=True)


# ---------------------------------------------------------------- setting up
def install(fast, cache, screen, colab):
    try:
        import qwen_tts  # noqa
        return
    except ImportError:
        pass
    wheels = os.path.join(cache, "wheels")
    if colab and not shutil.which("sox"):
        subprocess.run("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y sox libsox-dev >/dev/null 2>&1", shell=True)
    if fast and glob.glob(os.path.join(wheels, "qwen_tts*")):
        screen.show("⏳", "Installing the voice engine from your Drive (fast start)...")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", wheels, "qwen-tts"],
                           capture_output=True, text=True)
        if r.returncode == 0:
            return
    screen.show("⏳", "Installing the voice engine (about 1-2 minutes)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qwen-tts", "soundfile"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Couldn't install the voice engine: " + (r.stderr or r.stdout)[-300:])
    if fast:                                           # keep the installers in Drive for next time (in the background)
        def keep():
            os.makedirs(wheels, exist_ok=True)
            subprocess.run([sys.executable, "-m", "pip", "download", "-q", "-d", wheels, "qwen-tts", "soundfile"],
                           capture_output=True, text=True)
        threading.Thread(target=keep, daemon=True).start()


def model_path(name, fast, cache, screen):
    """A model folder: from your Drive when fast start has kept it there, else downloaded (and kept, if asked)."""
    local = os.path.join(cache, "models", name.replace("/", "__"))
    if fast and os.path.exists(os.path.join(local, "config.json")) and os.path.exists(os.path.join(local, ".complete")):
        return local
    from huggingface_hub import snapshot_download
    p = snapshot_download(name)
    if fast:
        def keep():
            try:
                tmp = local + ".part"
                shutil.rmtree(tmp, ignore_errors=True)
                shutil.copytree(p, tmp)
                shutil.rmtree(local, ignore_errors=True)
                os.replace(tmp, local)
                open(os.path.join(local, ".complete"), "w").close()
            except Exception:
                pass
        threading.Thread(target=keep, daemon=True).start()
    return p


# ---------------------------------------------------------------- the worker
class Worker:
    def __init__(self, root, device, colab, screen, base_model, gpu=""):
        self.root, self.device, self.colab, self.screen = root, device, colab, screen
        self.gpu = gpu
        self.base_name = base_model
        self.base = self.design = None
        self.prompts = {}
        self.state = {"state": "starting", "msg": "", "job": None, "done": 0, "total": 0}
        self.settings = _rj(os.path.join(root, "settings.json"), {}) or {}
        self.cache = os.path.join(root, "cache")
        for sub in ("jobs", "results", "voices"):
            os.makedirs(os.path.join(root, sub), exist_ok=True)
        threading.Thread(target=self._beat, daemon=True).start()

    def _beat(self):
        while True:
            try:
                _wj(os.path.join(self.root, "worker.json"), dict(self.state, t=time.time(), gpu=self.gpu, device=self.device,
                                                                 version=WORKER_VERSION, where="colab" if self.colab else "computer"))
            except Exception:
                pass
            time.sleep(HEARTBEAT)

    def set(self, state, msg, **kw):
        self.state.update(state=state, msg=msg, **kw)

    def _load(self, name):
        import torch
        from qwen_tts import Qwen3TTSModel
        fast = bool(self.settings.get("fast_start"))
        path = model_path(name, fast, self.cache, self.screen)
        if self.device == "cuda":
            return Qwen3TTSModel.from_pretrained(path, device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="eager")
        return Qwen3TTSModel.from_pretrained(path, device_map="cpu", dtype=torch.float32, attn_implementation="eager")

    def base_model(self):
        if self.base is None:
            self.set("loading", "Loading the cloning model...")
            self.screen.show("⏳", "Loading the voice model...")
            self.base = self._load(self.base_name)
        return self.base

    def design_model(self):
        if self.design is None:
            self.set("loading", "Loading the voice design model...")
            self.screen.show("⏳", "Loading the voice design model (the first design takes a minute)...")
            self.design = self._load(DESIGN)
        return self.design

    def prompt(self, vid):
        """The voice, prepared once from its sample and the words said in it."""
        vd = os.path.join(self.root, "voices", vid)
        meta = _rj(os.path.join(vd, "voice.json"), {}) or {}
        sample = os.path.join(vd, "sample.wav")
        if not os.path.exists(sample):
            raise RuntimeError(f"The voice '{meta.get('name') or vid}' hasn't reached this Drive yet. Wait a minute.")
        key = (vid, _md5(sample), meta.get("transcript"))
        if key not in self.prompts:
            m = self.base_model()
            text = (meta.get("transcript") or "").strip()
            if text:
                self.prompts[key] = m.create_voice_clone_prompt(ref_audio=sample, ref_text=text)
            else:                                     # no transcript: only the voice's colour (less close)
                self.prompts[key] = m.create_voice_clone_prompt(ref_audio=sample, ref_text=None, x_vector_only_mode=True)
        return self.prompts[key]

    def jobs(self):
        out = []
        for p in glob.glob(os.path.join(self.root, "jobs", "*.json")):
            j = _rj(p)
            if not j or not j.get("id"):
                continue
            rd = os.path.join(self.root, "results", j["id"])
            if os.path.exists(os.path.join(rd, "done")) or os.path.exists(os.path.join(rd, "error.txt")):
                continue
            out.append((j.get("created", 0), p, j))
        return [x[1:] for x in sorted(out)]

    def run_job(self, path, job):
        import numpy as np
        import torch
        rd = os.path.join(self.root, "results", job["id"])
        os.makedirs(rd, exist_ok=True)
        chunks = job.get("chunks") or []
        n = len(chunks)
        lang = job.get("language") or "Auto"
        kw = {"temperature": float(job.get("temperature", 0.8)), "max_new_tokens": int(job.get("max_new_tokens", 2048))}
        label = job.get("label") or "voice"
        t0, made = time.time(), 0
        for i, ch in enumerate(chunks):
            out = os.path.join(rd, f"{i:04d}.wav")
            if os.path.exists(out):
                continue
            if not os.path.exists(path):              # Video Studio cancelled this job
                self.set("ready", "Cancelled.")
                return
            self.set("working", f"Making voice: {label}, part {i + 1} of {n}", job=job["id"], done=i, total=n)
            el = time.time() - t0
            eta = f", about {int((el / made) * (n - i) / 60) + 1} min left" if made else ""
            self.screen.show("🎙️", f"Making voice: {label}, part {i + 1} of {n}{eta}",
                             "Keep this tab open. You can keep working in Video Studio.")
            torch.manual_seed(int(job.get("seed", 1234)) + i)
            if job.get("kind") == "design":
                wavs, sr = self.design_model().generate_voice_design(text=ch["text"], instruct=job.get("instruct") or "",
                                                                     language=lang, **kw)
            else:
                wavs, sr = self.base_model().generate_voice_clone(text=ch["text"], language=lang,
                                                                  voice_clone_prompt=self.prompt(job["voice"]), **kw)
            a = wavs[0]
            if hasattr(a, "cpu"):
                a = a.cpu().numpy()
            _write_wav(out, np.squeeze(np.asarray(a, dtype=np.float32)), sr)
            made += 1
        open(os.path.join(rd, "done"), "w").close()
        self.set("ready", f"Finished: {label}.", job=None, done=n, total=n)

    def loop(self, idle_exit=None):
        self.set("ready", "Ready.")
        self.screen.show("✅", f"Ready (voice engine v{WORKER_VERSION}). Go back to Video Studio.", "Keep this tab open while Video Studio makes voices.")
        idle = time.time()
        while True:
            self.settings = _rj(os.path.join(self.root, "settings.json"), {}) or self.settings
            todo = self.jobs()
            for path, job in todo:
                try:
                    self.run_job(path, job)
                except Exception as e:
                    rd = os.path.join(self.root, "results", job["id"])
                    os.makedirs(rd, exist_ok=True)
                    msg = str(e).strip().splitlines()[-1] if str(e).strip() else type(e).__name__
                    if "out of memory" in str(e).lower():
                        msg = "The graphics card ran out of memory. Runtime > Restart session, then Run again."
                    with open(os.path.join(rd, "error.txt"), "w", encoding="utf-8") as f:
                        f.write(msg[:500] + "\n\n" + traceback.format_exc()[-2000:])
                    self.set("ready", "Error: " + msg[:200])
                self.screen.show("✅", f"Ready (voice engine v{WORKER_VERSION}). Go back to Video Studio.", "Keep this tab open while Video Studio makes voices.")
                idle = time.time()
            if idle_exit and time.time() - idle > idle_exit:
                self.set("stopped", "Stopped (nothing to do).")
                _wj(os.path.join(self.root, "worker.json"), dict(self.state, t=0))
                return
            time.sleep(3)


# ---------------------------------------------------------------- the two ways to start it
def colab_main():
    screen = Screen(True)
    screen.show("⏳", f"Starting the Video Studio voice engine v{WORKER_VERSION}...")
    try:
        import torch
        if not torch.cuda.is_available():
            screen.show("⚠️", "This Colab has no graphics card turned on.",
                        "Click Runtime → Change runtime type → T4 GPU → Save, then press Run again.")
            return
        gpu = torch.cuda.get_device_name(0)
    except Exception as e:
        screen.show("⚠️", "Colab isn't ready: " + str(e)[:120])
        return
    screen.show("🔑", "Allow Google Drive in the window that opens...", "Video Studio and this notebook talk through your Drive.")
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:                            # the Google Drive permission didn't go through
        msg = str(e)
        screen.show("⚠️", "Google Drive wasn't allowed, so Video Studio can't talk to this notebook.",
                    "Fix: Runtime → Restart session, press ▶ Run again, choose the SAME Google account as this Colab "
                    "(top right), tick <b>Select all</b> on the permissions page, then Continue. No window? Allow pop-ups "
                    "for this site. Still failing? Open this notebook in an Incognito window with only that one account."
                    + (f"<br><span style='opacity:.6'>({msg[:120]})</span>" if msg else ""))
        return
    root = os.path.join("/content/drive/MyDrive", FOLDER)
    os.makedirs(root, exist_ok=True)
    settings = _rj(os.path.join(root, "settings.json"), {}) or {}
    fast = bool(settings.get("fast_start"))
    try:
        install(fast, os.path.join(root, "cache"), screen, True)
        w = Worker(root, "cuda", True, screen, BASE_GPU, gpu)
        w.base_model()
    except Exception as e:
        screen.show("⚠️", "Couldn't start: " + str(e)[:200], "Press Run again. If it happens again, Runtime → Disconnect and delete runtime, then Run.")
        raise
    w.loop()


def local_main(argv):
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True)
    ap.add_argument("--model", default=BASE_CPU)
    ap.add_argument("--idle-exit", type=float, default=600)
    a = ap.parse_args(argv)
    screen = Screen(False)
    dev = "cpu"
    try:
        import torch
        if torch.cuda.is_available():
            dev = "cuda"
    except Exception:
        pass
    w = Worker(a.root, dev, False, screen, a.model if dev == "cpu" else BASE_GPU,
               "this computer's processor" if dev == "cpu" else "this computer's graphics card")
    w.loop(idle_exit=a.idle_exit)


def notebook_cell():
    """The whole Colab cell: this file, then start it."""
    with open(os.path.abspath(__file__), encoding="utf-8") as f:
        src = f.read()
    src = src.split("\nif __name__ == \"__main__\"")[0]
    head = (f"#@title Video Studio voice engine v{WORKER_VERSION}: press ▶ (Run), then allow Google Drive\n"
            "# VIDEO STUDIO VOICE ENGINE - press Run (▶) and allow Google Drive. Nothing to change here.\n"
            "# Turn on the graphics card first: Runtime > Change runtime type > T4 GPU.\n")
    return head + src + "\n\ncolab_main()\n"


def notebook_json():
    return json.dumps({"nbformat": 4, "nbformat_minor": 0,
                       "metadata": {"accelerator": "GPU", "colab": {"provenance": [], "gpuType": "T4", "name": "Video Studio Voice"},
                                    "kernelspec": {"name": "python3", "display_name": "Python 3"}},
                       "cells": [{"cell_type": "code", "metadata": {"cellView": "form"}, "execution_count": None, "outputs": [],
                                  "source": notebook_cell().splitlines(True)}]}, indent=1)



colab_main()
